# 00 — A Graph Without an LLM

This first graph is deliberately deterministic. Its purpose is to show that Graph Engineering starts with state and control flow, not prompts or model calls.

You will define state, write nodes that return updates, connect edges, route on a condition, and inspect the execution trace. No API key is required.

## Topology

The classifier computes whether a value is even or odd. The router reads that category and selects one branch.

```mermaid
flowchart TD
    accTitle: Deterministic Number Graph
    accDescr: A value is classified as even or odd, then routed to multiplication or addition before the graph ends.

    start([START]) --> classify[Classify]
    classify --> router{Even?}
    router -->|yes| multiply[Multiply by two]
    router -->|no| add[Add three]
    multiply --> finish([END])
    add --> finish
```

## State and reducers

State is the shared execution contract. `trace` uses an append reducer because several nodes contribute events; current-value fields such as `category` and `result` use the runtime's normal overwrite behavior.

In [1]:
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class GraphState(TypedDict, total=False):
    value: int
    category: Literal["even", "odd"]
    result: int
    trace: Annotated[list[str], operator.add]

## Nodes compute

Each node reads state and returns only its explicit update. None mutates the input mapping or decides which function to call next.

In [2]:
def classify(state: GraphState) -> dict:
    category = "even" if state["value"] % 2 == 0 else "odd"
    return {"category": category, "trace": [f"classify:{category}"]}


def multiply(state: GraphState) -> dict:
    return {"result": state["value"] * 2, "trace": ["multiply"]}


def add(state: GraphState) -> dict:
    return {"result": state["value"] + 3, "trace": ["add"]}

## Routers control

The classifier produced a fact. The router maps that fact to a named edge. This separation is the principle: **nodes compute; routers control.**

In [3]:
def route_number(state: GraphState) -> Literal["even", "odd"]:
    category = state["category"]
    if category not in {"even", "odd"}:
        raise ValueError(f"Unsupported category: {category!r}")
    return category

## Edges and compilation

The builder makes topology explicit. Compilation validates and produces an executable graph; it does not run the graph yet.

In [4]:
builder = StateGraph(GraphState)
builder.add_node("classify", classify)
builder.add_node("multiply", multiply)
builder.add_node("add", add)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_number,
    {"even": "multiply", "odd": "add"},
)
builder.add_edge("multiply", END)
builder.add_edge("add", END)

graph = builder.compile()

## Invocation and execution trace

Invoke both routes. The assertions turn the examples into executable claims about control flow.

In [5]:
even_result = graph.invoke({"value": 6, "trace": []})
odd_result = graph.invoke({"value": 5, "trace": []})

assert even_result["result"] == 12
assert even_result["trace"] == ["classify:even", "multiply"]
assert odd_result["result"] == 8
assert odd_result["trace"] == ["classify:odd", "add"]

print(even_result)
print(odd_result)

{'value': 6, 'category': 'even', 'result': 12, 'trace': ['classify:even', 'multiply']}
{'value': 5, 'category': 'odd', 'result': 8, 'trace': ['classify:odd', 'add']}


In [6]:
for event in graph.stream({"value": 6, "trace": []}, stream_mode="updates"):
    print(event)

{'classify': {'category': 'even', 'trace': ['classify:even']}}
{'multiply': {'result': 12, 'trace': ['multiply']}}


## Conceptual observation

Replacing a deterministic node with an LLM does not fundamentally change the graph topology.

The LLM changes computation. The graph still owns control.